# Sliding Window Memory
> Keep the last N tokens of conversation, automatically evicting the oldest turns.

`SlidingWindowMemory` is the simplest conversation memory strategy: it keeps a
fixed-size window of recent turns and drops older ones when the token limit is
exceeded.

**Time:** ~5 minutes

## Setup

In [ ]:
from anchor.memory import SlidingWindowMemory

## Create a Sliding Window
We set a small token budget so we can observe eviction behavior quickly.

In [ ]:
memory = SlidingWindowMemory(max_tokens=2048)

print(f"Max tokens: {memory.max_tokens}")
print(f"Current turns: {len(memory.turns)}")
print(f"Current tokens: {memory.total_tokens}")

## Add Conversation Turns
Each turn can carry optional metadata (timestamps, tool IDs, etc.).

In [ ]:
# Add several rounds of conversation
memory.add_turn("user", "Hello!", metadata={"timestamp": "2024-01-01"})
memory.add_turn("assistant", "Hi there! How can I help?")
memory.add_turn("user", "What is the capital of France?")
memory.add_turn("assistant", "The capital of France is Paris.")

print(f"Turns: {len(memory.turns)}")
print(f"Total tokens: {memory.total_tokens}")

## Inspect Stored Turns
The `.turns` property exposes `ConversationTurn` objects.

In [ ]:
for i, turn in enumerate(memory.turns):
    meta = f"  meta={turn.metadata}" if turn.metadata else ""
    print(f"[{i}] {turn.role}: {turn.content[:60]}{meta}")

## Export as Context Items
`.as_context_items()` produces a `list[ContextItem]` ready for the pipeline.

In [ ]:
items = memory.as_context_items()

print(f"Context items produced: {len(items)}")
for item in items:
    print(f"  - {item.source_type.value}: {str(item.content)[:60]}")

## Observe Eviction
Push many turns to exceed `max_tokens` and watch the window slide.

In [ ]:
# Use a tiny window to force eviction
small = SlidingWindowMemory(max_tokens=150)

for i in range(10):
    small.add_turn("user", f"Message number {i} with some padding text.")
    small.add_turn("assistant", f"Reply to message {i}.")

print(f"Turns remaining: {len(small.turns)}")
print(f"Tokens used: {small.total_tokens} / {small.max_tokens}")
print("\nSurviving turns:")
for turn in small.turns:
    print(f"  {turn.role}: {turn.content}")

## Key Takeaways
- `SlidingWindowMemory` maintains a FIFO window capped by token count
- `.add_turn()` accepts `role`, `content`, and optional `metadata`
- `.as_context_items()` bridges memory to the context pipeline
- Oldest turns are evicted first when the budget is exceeded

**Next:** [Summary Buffer](02_summary_buffer.ipynb)